# contract-resolver — use cases

`contract-resolver` turns a **declarative description** of an option structure
into **concrete, identified contracts**. This notebook runs entirely **offline**
against a synthetic chain (a `StaticSource`) — no credentials, no network — then
shows the one-line swap to go **live** with `BrokerSource`.

Pipeline:  **description → select_contract → identity → ResolvedLeg**, composed by
`StructureResolver`.


In [1]:
import math
from datetime import date
from decimal import Decimal

import pandas as pd

from contract_resolver import (
    ExpirationSpec, OptionStructure, StrikeSelection, StructureLeg,
    StructureResolver, select_contract,
)
from contract_resolver.models import ContractRow
from contract_resolver.sources import StaticSource

# A real trading-day expiration. (2026-07-03 is the observed Independence Day
# holiday — markets closed — so nothing lists there; 07-02 is a normal Thursday.)
EXP, SPOT = date(2026, 7, 2), 6300

def _occ(cp, strike):
    return f"SPXW{EXP:%y%m%d}{cp}{int(strike) * 1000:08d}"

def make_chain():
    """A synthetic SPXW chain on a 25-pt grid with smooth deltas + premiums."""
    rows = []
    for k in range(6000, 6601, 25):
        call_delta = max(0.01, min(0.99, 0.5 - (k - SPOT) / 400))
        tv = 50 * math.exp(-((k - SPOT) / 200) ** 2)        # time value bump
        rows.append(ContractRow("call", Decimal(k), _occ("C", k), expiration=EXP,
                                delta=Decimal(str(round(call_delta, 3))),
                                mid=Decimal(str(round(max(0, SPOT - k) + tv, 2)))))
        rows.append(ContractRow("put", Decimal(k), _occ("P", k), expiration=EXP,
                                delta=Decimal(str(round(call_delta - 1, 3))),
                                mid=Decimal(str(round(max(0, k - SPOT) + tv, 2)))))
    return rows

ROWS = make_chain()
CALLS = [r for r in ROWS if r.option_type == "call"]
PUTS = [r for r in ROWS if r.option_type == "put"]
source = StaticSource(ROWS)
print(f"synthetic SPXW chain: {len(ROWS)} contracts ({len(CALLS)} calls / {len(PUTS)} puts), spot {SPOT}")

def show(struct, res):
    """Render a StructureResolution as a DataFrame (legs or failures)."""
    if not res.ok:
        return pd.DataFrame([{"role": f.role, "status": "NoMatch", "reason": f.reason.value} for f in res.failures])
    by_role = {leg.role: leg for leg in struct.legs}
    out = []
    for role, leg in res.resolved.legs.items():
        sl = by_role[role]
        out.append({"role": role, "side": sl.side, "type": sl.option_type,
                    "strike": float(leg.strike),
                    "delta": round(float(leg.delta), 3) if leg.delta is not None else None,
                    "mid": float(leg.mid) if leg.mid is not None else None,
                    "security_id": leg.security_id})
    return pd.DataFrame(out)

synthetic SPXW chain: 50 contracts (25 calls / 25 puts), spot 6300


> **On expirations & the trading calendar.** This synthetic chain has a single,
> real trading-day expiration (`2026-07-02`). It deliberately avoids `2026-07-03`,
> which is the *observed* Independence Day holiday (markets closed) — the kind of
> date that must never appear as a listed contract. With `StaticSource` you own the
> contracts, so `dte` here is illustrative. **Live `BrokerSource` can't pick a
> non-existent date:** it matches `ExpirationSpec` against the *actual* expirations
> the TastyTrade chain returns — the chain is the source of truth, so closed days
> simply aren't candidates.

## 1. Author a structure (the description)

Pure, declarative — this is what a bot writes / what's stored as config. An iron
condor: short strikes by **delta**, wings by **offset_from** the short.

In [2]:
def iron_condor(short_delta=0.16, wing=25, dte_spec=ExpirationSpec(dte=3)):
    return OptionStructure(
        underlying="SPX", expiration=dte_spec,
        legs=(
            StructureLeg("short_put",  "put",  "short", StrikeSelection("delta", target=short_delta)),
            StructureLeg("long_put",   "put",  "long",  StrikeSelection("offset_from", role="short_put",  offset=-wing)),
            StructureLeg("short_call", "call", "short", StrikeSelection("delta", target=short_delta)),
            StructureLeg("long_call",  "call", "long",  StrikeSelection("offset_from", role="short_call", offset=wing)),
        ),
    )

ic = iron_condor()
pd.DataFrame([{"role": leg.role, "side": leg.side, "type": leg.option_type,
               "selection": leg.strike_selection.method,
               "target/offset": leg.strike_selection.target if leg.strike_selection.target is not None else leg.strike_selection.offset}
              for leg in ic.legs])

,role,side,type,selection,target/offset
0,short_put,short,put,delta,0.16
1,long_put,long,put,offset_from,-25.00
2,short_call,short,call,delta,0.16
3,long_call,long,call,offset_from,25.00


## 2. Resolve it  (`StructureResolver` + `StaticSource`)

The shorts resolve by delta; the wings resolve `offset_from` the shorts (two-pass
ordering). Each leg comes back identified — a `security_id`.

In [3]:
res = await StructureResolver(source).resolve(ic)
print("ok:", res.ok)
show(ic, res)

ok: True


,role,side,type,strike,delta,mid,security_id
0,short_put,short,put,6175.0,-0.188,33.83,option:SPXW:2026-07-02:put:6175
1,short_call,short,call,6425.0,0.188,33.83,option:SPXW:2026-07-02:call:6425
2,long_put,long,put,6150.0,-0.125,28.49,option:SPXW:2026-07-02:put:6150
3,long_call,long,call,6450.0,0.125,28.49,option:SPXW:2026-07-02:call:6450


## 3. The six selection methods  (`select_contract`, pure)

The one place strikes get picked — works on any candidate list, no I/O.

In [4]:
examples = [
    ("delta 0.30 (call)",        CALLS, StrikeSelection("delta", target=0.30), {}),
    ("strike 6300 (call)",       CALLS, StrikeSelection("strike", target=6300), {}),
    ("premium ~15 (put)",        PUTS,  StrikeSelection("premium", target=15), {}),
    ("atm_offset +50 (call)",    CALLS, StrikeSelection("atm_offset", offset=50), {}),
    ("offset_from 6425 +25",     CALLS, StrikeSelection("offset_from", role="s", offset=25), {"s": Decimal(6425)}),
    ("absolute_price 6312 (call)", CALLS, StrikeSelection("absolute_price", target=6312), {}),
]
rows = []
for label, cands, sel, sib in examples:
    c = select_contract(cands, sel, siblings=sib)
    rows.append({"method": label, "picked strike": float(c.strike) if c else None,
                 "delta": float(c.delta) if c and c.delta is not None else None,
                 "mid": float(c.mid) if c and c.mid is not None else None})
pd.DataFrame(rows)

,method,picked strike,delta,mid
0,delta 0.30 (call),6375.0,0.312,43.44
1,strike 6300 (call),6300.0,0.500,50.00
2,premium ~15 (put),6075.0,-0.010,14.10
3,atm_offset +50 (call),6350.0,0.375,46.97
4,offset_from 6425 +25,6450.0,0.125,28.49
5,absolute_price 6312 (call),6300.0,0.500,50.00


## 4. A gallery of structures

Same machinery, different descriptions — the vocabulary covers the common
defined-risk and premium-targeted shapes.

In [5]:
async def resolve(struct):
    return show(struct, await StructureResolver(source).resolve(struct))

# put credit spread — short by delta, long wing by offset
pcs = OptionStructure("SPX", expiration=ExpirationSpec(dte=3), legs=(
    StructureLeg("short_put", "put", "short", StrikeSelection("delta", target=0.30)),
    StructureLeg("long_put",  "put", "long",  StrikeSelection("offset_from", role="short_put", offset=-50)),
))
print("PUT CREDIT SPREAD")
display(await resolve(pcs))

# iron butterfly — ATM short put, co-located short call, wings by offset
ibf = OptionStructure("SPX", expiration=ExpirationSpec(dte=3), legs=(
    StructureLeg("short_put",  "put",  "short", StrikeSelection("atm_offset", offset=0)),
    StructureLeg("short_call", "call", "short", StrikeSelection("offset_from", role="short_put", offset=0)),
    StructureLeg("long_put",   "put",  "long",  StrikeSelection("offset_from", role="short_put", offset=-50)),
    StructureLeg("long_call",  "call", "long",  StrikeSelection("offset_from", role="short_put", offset=50)),
))
print("IRON BUTTERFLY")
display(await resolve(ibf))

# MEIC-style — shorts by TARGET PREMIUM, wings by offset
meic = OptionStructure("SPX", expiration=ExpirationSpec(dte=3), legs=(
    StructureLeg("short_put",  "put",  "short", StrikeSelection("premium", target=12)),
    StructureLeg("long_put",   "put",  "long",  StrikeSelection("offset_from", role="short_put", offset=-25)),
    StructureLeg("short_call", "call", "short", StrikeSelection("premium", target=12)),
    StructureLeg("long_call",  "call", "long",  StrikeSelection("offset_from", role="short_call", offset=25)),
))
print("MEIC (shorts by premium)")
display(await resolve(meic))

PUT CREDIT SPREAD


,role,side,type,strike,delta,mid,security_id
0,short_put,short,put,6225.0,-0.312,43.44,option:SPXW:2026-07-02:put:6225
1,long_put,long,put,6175.0,-0.188,33.83,option:SPXW:2026-07-02:put:6175


IRON BUTTERFLY


,role,side,type,strike,delta,mid,security_id
0,short_put,short,put,6300.0,-0.500,50.00,option:SPXW:2026-07-02:put:6300
1,short_call,short,call,6300.0,0.500,50.00,option:SPXW:2026-07-02:call:6300
2,long_put,long,put,6250.0,-0.375,46.97,option:SPXW:2026-07-02:put:6250
3,long_call,long,call,6350.0,0.375,46.97,option:SPXW:2026-07-02:call:6350


MEIC (shorts by premium)


,role,side,type,strike,delta,mid,security_id
0,short_put,short,put,6050.0,-0.01,10.48,option:SPXW:2026-07-02:put:6050
1,short_call,short,call,6550.0,0.01,10.48,option:SPXW:2026-07-02:call:6550
2,long_put,long,put,6025.0,-0.01,7.55,option:SPXW:2026-07-02:put:6025
3,long_call,long,call,6575.0,0.01,7.55,option:SPXW:2026-07-02:call:6575


## 5. Options on futures  (`/ES`)

The **same** description machinery resolves options-on-futures — only the
underlying (`/ES`) and the resulting identity differ. Legs come back as
`future_option:…` ids that **carry the opaque `order_symbol`** (the CME series
code) plus physical settlement and the contract multiplier. (Live `BrokerSource`
resolves `/ES` against the real front-month future + product codes from the TT
futures-option chain; here the chain is synthetic.)

In [6]:
ES_SPOT, ES_EXP = 6300, date(2026, 7, 1)   # 2026-07-01 is a Wednesday — a real ES weekly expiry

def es_chain():
    rows = []
    for k in range(6000, 6601, 25):
        cd = max(0.01, min(0.99, 0.5 - (k - ES_SPOT) / 400))
        tv = 55 * math.exp(-((k - ES_SPOT) / 200) ** 2)
        # TT futures-option order symbol: ./<future> <series> <yymmdd><C|P><strike>
        c_sym = f"./ESU6 E1CN6 {ES_EXP:%y%m%d}C{k}"
        p_sym = f"./ESU6 E1CN6 {ES_EXP:%y%m%d}P{k}"
        rows.append(ContractRow("call", Decimal(k), c_sym, expiration=ES_EXP,
                                delta=Decimal(str(round(cd, 3))), mid=Decimal(str(round(max(0, ES_SPOT - k) + tv, 2)))))
        rows.append(ContractRow("put", Decimal(k), p_sym, expiration=ES_EXP,
                                delta=Decimal(str(round(cd - 1, 3))), mid=Decimal(str(round(max(0, k - ES_SPOT) + tv, 2)))))
    return rows

es_source = StaticSource(es_chain())

def show_fut(struct, res):
    by_role = {leg.role: leg for leg in struct.legs}
    out = []
    for role, leg in res.resolved.legs.items():
        sl, sec = by_role[role], leg.security
        out.append({"role": role, "side": sl.side, "strike": float(leg.strike),
                    "delta": round(float(leg.delta), 3) if leg.delta is not None else None,
                    "security_id": sec.security_id, "order_symbol": sec.order_symbol,
                    "settle": str(sec.settlement_type),
                    "x": float(sec.contract_multiplier) if sec.contract_multiplier else None})
    return pd.DataFrame(out)

es_ic = OptionStructure("/ES", expiration=ExpirationSpec(dte=1), legs=(
    StructureLeg("short_put",  "put",  "short", StrikeSelection("delta", target=0.16)),
    StructureLeg("long_put",   "put",  "long",  StrikeSelection("offset_from", role="short_put",  offset=-50)),
    StructureLeg("short_call", "call", "short", StrikeSelection("delta", target=0.16)),
    StructureLeg("long_call",  "call", "long",  StrikeSelection("offset_from", role="short_call", offset=50)),
))
res_es = await StructureResolver(es_source).resolve(es_ic)
print("ES futures-option iron condor — ok:", res_es.ok)
show_fut(es_ic, res_es)

ES futures-option iron condor — ok: True


,role,side,strike,delta,security_id,order_symbol,settle,x
0,short_put,short,6175.0,-0.188,future_option:ES:U6:2026-07-01:put:6175,./ESU6 E1CN6 260701P6175,physical,50.0
1,short_call,short,6425.0,0.188,future_option:ES:U6:2026-07-01:call:6425,./ESU6 E1CN6 260701C6425,physical,50.0
2,long_put,long,6125.0,-0.062,future_option:ES:U6:2026-07-01:put:6125,./ESU6 E1CN6 260701P6125,physical,50.0
3,long_call,long,6475.0,0.062,future_option:ES:U6:2026-07-01:call:6475,./ESU6 E1CN6 260701C6475,physical,50.0


## 6. Identity — `security_id` & `order_symbol`

Every resolved leg is a `security-universe` `Security`. Equity/index options get
an algorithmic id; **options-on-futures also carry the opaque `order_symbol`** so
the broker symbol round-trips off the object (no registry).

In [7]:
from contract_resolver.identity import to_resolved_leg
from security_universe.resolvers import ResolverChain

eq = res.resolved.legs["short_put"].security          # from the iron condor above
print("equity/index option")
print("  security_id  :", eq.security_id)
print("  order_symbol :", eq.order_symbol, "(algorithmic from id)")
print("  settlement   :", eq.settlement_type)

fo_row = ContractRow("call", Decimal(6325), "./ESU6 E1CN6 260701C6325", expiration=date(2026, 7, 1))
fo = to_resolved_leg(fo_row, ResolverChain.default(), "demo", None).security
print("\noption on future")
print("  security_id  :", fo.security_id)
print("  order_symbol :", fo.order_symbol, "(opaque series code carried)")
print("  settlement   :", fo.settlement_type, "· x", fo.contract_multiplier)

equity/index option
  security_id  : option:SPXW:2026-07-02:put:6175
  order_symbol : None (algorithmic from id)
  settlement   : cash

option on future
  security_id  : future_option:ES:U6:2026-07-01:call:6325
  order_symbol : ./ESU6 E1CN6 260701C6325 (opaque series code carried)
  settlement   : physical · x 50


## 7. Failure handling — typed outcomes

A miss is never a bare `None`; it's a typed `NoMatch(reason, role)`. Here an
exact `strike` that isn't on the grid fails, and its `offset_from` child can't
resolve either.

In [8]:
bad = OptionStructure("SPX", expiration=ExpirationSpec(dte=3), legs=(
    StructureLeg("short_put", "put", "short", StrikeSelection("strike", target=6312)),   # off-grid → miss
    StructureLeg("long_put",  "put", "long",  StrikeSelection("offset_from", role="short_put", offset=-25)),
))
show(bad, await StructureResolver(source).resolve(bad))

,role,status,reason
0,short_put,NoMatch,no_strike_in_tolerance
1,long_put,NoMatch,sibling_unresolved


## 8. Going live — swap `StaticSource` for `BrokerSource`

Nothing above changes. `BrokerSource` resolves the *same* descriptions against
live TastyTrade (REST chain ⊕ DXLink greeks) — strike/offset selection is pure
REST; delta/premium add a bounded greeks subscription. Requires the `broker`
extra + `TASTYTRADE_*` creds in `.env`.

```python
from dxlink_client.providers.tastytrade import TastytradeTokenProvider
from contract_resolver.sources.broker import BrokerSource
from contract_resolver import StructureResolver

source = BrokerSource(TastytradeTokenProvider())      # reads TASTYTRADE_* from .env
res = await StructureResolver(source).resolve(ic)      # same call, live data
#   res.resolved.legs[...].security.order_symbol  → the TastyTrade order symbol
```

Or a single leg, e.g. the live `/ES` example from the docs:

```python
leg = await source.resolve_leg(
    "/ES", ExpirationSpec(dte=1), "call", StrikeSelection("delta", target=0.30),
)
#   → future_option:ES:U6:2026-07-01:call:6325  ·  ./ESU6 E1CN6 260701C6325
```


---
**Takeaways**

- One declarative `OptionStructure` resolves the same way offline (`StaticSource`)
  or live (`BrokerSource`) — the description never changes.
- **Options on futures** go through the same path: `/ES` → `future_option:…` with
  the opaque `order_symbol` carried and physical settlement.
- `select_contract` is the single, pure strike picker (six methods).
- `StructureResolver` composes legs with `offset_from` ordering + cycle detection.
- Every leg is identified (`security_id`); options-on-futures also carry `order_symbol`.
- Misses are typed `NoMatch` outcomes, not exceptions.
